In [ ]:
GREEDY

In [ ]:
import tkinter as tk
from tkinter import messagebox, scrolledtext
import heapq
import time
import random

class Node:
    def __init__(self, state, parent=None, action="START", g_cost=0, h_cost=0, node_id="A"):
        self.state = state     # Tuple 2D 3x3
        self.parent = parent   
        self.action = action   # "Up", "Down", "Left", "Right" hoặc "START"
        self.g = g_cost        
        self.h = h_cost        
        self.f = g_cost + h_cost
        self.id = node_id      # Định danh Node (A, B, C...) để in Log giống mẫu

    def __lt__(self, other):
        return self.f < other.f

goal_puzzle = ((1, 2, 3), (4, 5, 6), (7, 8, 0))
node_counter = 65 # Mã ASCII để sinh tên Node từ 'A', 'B', 'C'...

def find_blank(state):
    for r in range(3):
        for c in range(3):
            if state[r][c] == 0: return r, c

def expand_8puzzle(node):
    global node_counter
    children = []
    r, c = find_blank(node.state)
    # Định nghĩa hướng di chuyển của ô trống và nhãn hành động tương ứng
    moves = [(-1, 0, "Up"), (1, 0, "Down"), (0, -1, "Left"), (0, 1, "Right")]
    
    for dr, dc, act in moves:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            state_list = [list(row) for row in node.state]
            state_list[r][c], state_list[nr][nc] = state_list[nr][nc], state_list[r][c]
            child_state = tuple(tuple(row) for row in state_list)
            
            node_counter += 1
            if node_counter > 90: node_counter = 65 # Reset vòng lại nếu vượt quá 'Z'
            c_id = chr(node_counter)
            
            children.append(Node(child_state, parent=node, action=act, g_cost=node.g + 1, node_id=c_id))
    return children

def generate_random_start(goal, steps=15):
    current = Node(goal)
    for _ in range(steps):
        children = expand_8puzzle(current)
        if children: current = random.choice(children)
    return current.state

def manhattan_heuristic_8puzzle(state, goal_state):
    goal_pos = {}
    for r in range(3):
        for c in range(3): goal_pos[goal_state[r][c]] = (r, c)
    distance = 0
    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0:
                g_r, g_c = goal_pos[val]
                distance += abs(r - g_r) + abs(c - g_c)
    return distance

def solution(node):
    path = []
    while node:
        path.append((node.state, node.action, node.id))
        node = node.parent
    return path[::-1]

class GreedyPuzzleGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("8-Puzzle Search Algorithms - GREEDY")
        self.root.geometry("1100x650")
        self.start_puzzle = generate_random_start(goal_puzzle)
        self.buttons = {}
        self.create_layout()

    def create_layout(self):
        # LEFT PANEL (Giao diện bảng số)
        left_frame = tk.Frame(self.root, width=350, padx=10, pady=10)
        left_frame.pack(side=tk.LEFT, fill=tk.Y)

        title_lbl = tk.Label(left_frame, text="8-Puzzle Search Algorithms", font=("Arial", 16, "bold"))
        title_lbl.pack(pady=10)

        self.grid_frame = tk.Frame(left_frame, bg="gray", bd=4)
        self.grid_frame.pack(pady=10)

        for r in range(3):
            for c in range(3):
                btn = tk.Label(self.grid_frame, text="", width=5, height=2, font=("Arial", 22, "bold"), relief="raised", bd=2)
                btn.grid(row=r, column=c, padx=3, pady=3)
                self.buttons[(r, c)] = btn
        
        self.update_grid(self.start_puzzle)

        # RIGHT PANEL (Thông số + Log Panel)
        right_frame = tk.Frame(self.root, padx=10, pady=10)
        right_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)

        # Thanh nút trên cùng bên phải
        top_bar = tk.Frame(right_frame)
        top_bar.pack(fill=tk.X, pady=5)
        
        tk.Label(top_bar, text="Algorithm: GREEDY", font=("Arial", 11, "bold")).pack(side=tk.LEFT, padx=5)
        tk.Button(top_bar, text="Random", bg="orange", command=self.trigger_random).pack(side=tk.RIGHT, padx=5)
        tk.Button(top_bar, text="Run", bg="lightgreen", command=self.solve_greedy).pack(side=tk.RIGHT, padx=5)
        tk.Button(top_bar, text="Reset", bg="lightgray", command=self.reset_game).pack(side=tk.RIGHT, padx=5)

        # Khu vực hiển thị thông số Metrics giống mẫu bài của bạn
        self.metrics_frame = tk.Frame(right_frame, pady=10)
        self.metrics_frame.pack(fill=tk.X)
        
        self.lbl_step = self.create_metric_line("Step:", "0")
        self.lbl_expanded = self.create_metric_line("Expanded:", "None")
        self.lbl_action = self.create_metric_line("Action:", "None")
        self.lbl_path_cost = self.create_metric_line("Path cost:", "0")
        self.lbl_frontier_size = self.create_metric_line("Frontier:", "0 node(s)")
        self.lbl_explored_size = self.create_metric_line("Explored:", "0 state(s)")

        # Log Panel (Text Area) có thanh cuộn dọc giống hệt ảnh mẫu
        tk.Label(right_frame, text="Children  Frontier  Explored  Step Log", font=("Arial", 11, "bold")).pack(anchor=tk.W, pady=(10,2))
        self.log_panel = scrolledtext.ScrolledText(right_frame, font=("Courier New", 10), bg="#f8f9fa", bd=3, relief="sunken")
        self.log_panel.pack(fill=tk.BOTH, expand=True)

    def create_metric_line(self, label_text, default_val):
        row = tk.Frame(self.metrics_frame)
        row.pack(anchor=tk.W, pady=1)
        tk.Label(row, text=label_text, width=12, anchor=tk.W, font=("Arial", 10, "bold")).pack(side=tk.LEFT)
        val_lbl = tk.Label(row, text=default_val, font=("Arial", 10))
        val_lbl.pack(side=tk.LEFT)
        return val_lbl

    def update_grid(self, state):
        for r in range(3):
            for c in range(3):
                val = state[r][c]
                self.buttons[(r, c)].config(text="" if val == 0 else str(val), bg="#2196F3" if val != 0 else "lightgray", fg="white" if val != 0 else "black")
        self.root.update()

    def trigger_random(self):
        self.start_puzzle = generate_random_start(goal_puzzle)
        self.update_grid(self.start_puzzle)
        self.log_panel.delete('1.0', tk.END)

    def reset_game(self):
        self.update_grid(self.start_puzzle)
        self.lbl_step.config(text="0")
        self.lbl_expanded.config(text="None")
        self.lbl_action.config(text="None")
        self.lbl_path_cost.config(text="0")
        self.lbl_frontier_size.config(text="0 node(s)")
        self.lbl_explored_size.config(text="0 state(s)")
        self.log_panel.delete('1.0', tk.END)

    def log(self, msg):
        self.log_panel.insert(tk.END, msg + "\n")
        self.log_panel.see(tk.END)

    def solve_greedy(self):
        global node_counter
        node_counter = 65 
        self.log_panel.delete('1.0', tk.END)
        self.log("--- GREEDY BEST-FIRST SEARCH EXECUTION LOG ---")

        h_start = manhattan_heuristic_8puzzle(self.start_puzzle, goal_puzzle)
        start_node = Node(self.start_puzzle, g_cost=0, h_cost=h_start, node_id="A")
        
        frontier = [(start_node.h, start_node)]
        reached = {self.start_puzzle: start_node}
        
        step_idx = 0
        
        while frontier:
            step_idx += 1
            _, n = heapq.heappop(frontier)
            
            # Kháng cập nhật thông số lên UI
            self.lbl_step.config(text=str(step_idx))
            self.lbl_expanded.config(text=n.id)
            self.lbl_action.config(text=n.action)
            self.lbl_path_cost.config(text=str(n.g))
            self.lbl_frontier_size.config(text=f"{len(frontier)} node(s)")
            self.lbl_explored_size.config(text=f"{len(reached)} state(s)")
            
            self.update_grid(n.state)
            time.sleep(0.4) # Chờ mượt để xem hoạt họa hình ảnh

            # Ghi Log chi tiết giống mẫu ảnh
            self.log(f"\n[STEP {step_idx}]")
            self.log(f"Chosen Node: Node ID {n.id} | Action: {n.action} | h(n) = {n.h}")
            
            if n.state == goal_puzzle:
                self.log(f"[STEP LOG END] -> Goal Found at Node {n.id}!")
                path = solution(n)
                messagebox.showinfo("Thành công", f"Greedy đã giải xong sau {len(path)-1} bước!")
                return

            self.log("Children Nodes generated and pushed to Frontier:")
            for m_node in expand_8puzzle(n):
                m = m_node.state
                g_new = m_node.g
                h_m = manhattan_heuristic_8puzzle(m, goal_puzzle)
                
                # Logic so sánh cập nhật chuẩn slide bài học
                if m in reached:
                    if g_new < reached[m].g:
                        reached[m].g = g_new
                        reached[m].parent = n
                        heapq.heappush(frontier, (h_m, reached[m]))
                        self.log(f"  - Node ID {m_node.id} | Action: {m_node.action} | Path Cost (g): {g_new} | Heuristic (h): {h_m} (Updated cheaper path)")
                else:
                    m_node.h = h_m
                    reached[m] = m_node
                    heapq.heappush(frontier, (h_m, m_node))
                    self.log(f"  - Node ID {m_node.id} | Action: {m_node.action} | Path Cost (g): {g_new} | Heuristic (h): {h_m}")
                    
        messagebox.showerror("Thất bại", "Không tìm thấy lời giải!")

if __name__ == "__main__":
    root = tk.Tk()
    app = GreedyPuzzleGUI(root)
    root.mainloop()

In [ ]:
A*

In [ ]:
import tkinter as tk
from tkinter import messagebox, scrolledtext
import heapq
import time
import random

class Node:
    def __init__(self, state, parent=None, action="START", g_cost=0, h_cost=0, node_id="A"):
        self.state = state     
        self.parent = parent   
        self.action = action   
        self.g = g_cost        
        self.h = h_cost        
        self.f = g_cost + h_cost
        self.id = node_id      

    def __lt__(self, other):
        return self.f < other.f

goal_puzzle = ((1, 2, 3), (4, 5, 6), (7, 8, 0))
node_counter = 65 

def find_blank(state):
    for r in range(3):
        for c in range(3):
            if state[r][c] == 0: return r, c

def expand_8puzzle(node):
    global node_counter
    children = []
    r, c = find_blank(node.state)
    moves = [(-1, 0, "Up"), (1, 0, "Down"), (0, -1, "Left"), (0, 1, "Right")]
    
    for dr, dc, act in moves:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            state_list = [list(row) for row in node.state]
            state_list[r][c], state_list[nr][nc] = state_list[nr][nc], state_list[r][c]
            child_state = tuple(tuple(row) for row in state_list)
            
            node_counter += 1
            if node_counter > 90: node_counter = 65 
            c_id = chr(node_counter)
            
            children.append(Node(child_state, parent=node, action=act, g_cost=node.g + 1, node_id=c_id))
    return children

def generate_random_start(goal, steps=15):
    current = Node(goal)
    for _ in range(steps):
        children = expand_8puzzle(current)
        if children: current = random.choice(children)
    return current.state

def manhattan_heuristic_8puzzle(state, goal_state):
    goal_pos = {}
    for r in range(3):
        for c in range(3): goal_pos[goal_state[r][c]] = (r, c)
    distance = 0
    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0:
                g_r, g_c = goal_pos[val]
                distance += abs(r - g_r) + abs(c - g_c)
    return distance

def solution(node):
    path = []
    while node:
        path.append((node.state, node.action, node.id))
        node = node.parent
    return path[::-1]

class AStarPuzzleGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("8-Puzzle Search Algorithms - A*")
        self.root.geometry("1100x650")
        self.start_puzzle = generate_random_start(goal_puzzle)
        self.buttons = {}
        self.create_layout()

    def create_layout(self):
        # LEFT PANEL
        left_frame = tk.Frame(self.root, width=350, padx=10, pady=10)
        left_frame.pack(side=tk.LEFT, fill=tk.Y)

        title_lbl = tk.Label(left_frame, text="8-Puzzle Search Algorithms", font=("Arial", 16, "bold"))
        title_lbl.pack(pady=10)

        self.grid_frame = tk.Frame(left_frame, bg="gray", bd=4)
        self.grid_frame.pack(pady=10)

        for r in range(3):
            for c in range(3):
                btn = tk.Label(self.grid_frame, text="", width=5, height=2, font=("Arial", 22, "bold"), relief="raised", bd=2)
                btn.grid(row=r, column=c, padx=3, pady=3)
                self.buttons[(r, c)] = btn
        
        self.update_grid(self.start_puzzle)

        # RIGHT PANEL
        right_frame = tk.Frame(self.root, padx=10, pady=10)
        right_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)

        top_bar = tk.Frame(right_frame)
        top_bar.pack(fill=tk.X, pady=5)
        
        tk.Label(top_bar, text="Algorithm: A*", font=("Arial", 11, "bold")).pack(side=tk.LEFT, padx=5)
        tk.Button(top_bar, text="Random", bg="orange", command=self.trigger_random).pack(side=tk.RIGHT, padx=5)
        tk.Button(top_bar, text="Run", bg="lightgreen", command=self.solve_astar).pack(side=tk.RIGHT, padx=5)
        tk.Button(top_bar, text="Reset", bg="lightgray", command=self.reset_game).pack(side=tk.RIGHT, padx=5)

        self.metrics_frame = tk.Frame(right_frame, pady=10)
        self.metrics_frame.pack(fill=tk.X)
        
        self.lbl_step = self.create_metric_line("Step:", "0")
        self.lbl_expanded = self.create_metric_line("Expanded:", "None")
        self.lbl_action = self.create_metric_line("Action:", "None")
        self.lbl_path_cost = self.create_metric_line("Path cost:", "0")
        self.lbl_frontier_size = self.create_metric_line("Frontier:", "0 node(s)")
        self.lbl_explored_size = self.create_metric_line("Explored:", "0 state(s)")

        tk.Label(right_frame, text="Children  Frontier  Explored  Step Log", font=("Arial", 11, "bold")).pack(anchor=tk.W, pady=(10,2))
        self.log_panel = scrolledtext.ScrolledText(right_frame, font=("Courier New", 10), bg="#f8f9fa", bd=3, relief="sunken")
        self.log_panel.pack(fill=tk.BOTH, expand=True)

    def create_metric_line(self, label_text, default_val):
        row = tk.Frame(self.metrics_frame)
        row.pack(anchor=tk.W, pady=1)
        tk.Label(row, text=label_text, width=12, anchor=tk.W, font=("Arial", 10, "bold")).pack(side=tk.LEFT)
        val_lbl = tk.Label(row, text=default_val, font=("Arial", 10))
        val_lbl.pack(side=tk.LEFT)
        return val_lbl

    def update_grid(self, state):
        for r in range(3):
            for c in range(3):
                val = state[r][c]
                self.buttons[(r, c)].config(text="" if val == 0 else str(val), bg="#2196F3" if val != 0 else "lightgray", fg="white" if val != 0 else "black")
        self.root.update()

    def trigger_random(self):
        self.start_puzzle = generate_random_start(goal_puzzle)
        self.update_grid(self.start_puzzle)
        self.log_panel.delete('1.0', tk.END)

    def reset_game(self):
        self.update_grid(self.start_puzzle)
        self.lbl_step.config(text="0")
        self.lbl_expanded.config(text="None")
        self.lbl_action.config(text="None")
        self.lbl_path_cost.config(text="0")
        self.lbl_frontier_size.config(text="0 node(s)")
        self.lbl_explored_size.config(text="0 state(s)")
        self.log_panel.delete('1.0', tk.END)

    def log(self, msg):
        self.log_panel.insert(tk.END, msg + "\n")
        self.log_panel.see(tk.END)

    def solve_astar(self):
        global node_counter
        node_counter = 65 
        self.log_panel.delete('1.0', tk.END)
        self.log("--- A* SEARCH ALGORITHM EXECUTION LOG ---")

        # 1 + 2. Khởi tạo theo slide
        h_start = manhattan_heuristic_8puzzle(self.start_puzzle, goal_puzzle)
        start_node = Node(self.start_puzzle, g_cost=0, h_cost=h_start, node_id="A")
        
        frontier = [(start_node.f, start_node)]
        reached = {self.start_puzzle: start_node} 
        
        step_idx = 0
        
        # 3. Vòng lặp chính chuẩn slide của cô
        while frontier:
            step_idx += 1
            _, n = heapq.heappop(frontier)
            
            self.lbl_step.config(text=str(step_idx))
            self.lbl_expanded.config(text=n.id)
            self.lbl_action.config(text=n.action)
            self.lbl_path_cost.config(text=str(n.g))
            self.lbl_frontier_size.config(text=f"{len(frontier)} node(s)")
            self.lbl_explored_size.config(text=f"{len(reached)} state(s)")
            
            self.update_grid(n.state)
            time.sleep(0.4)

            self.log(f"\n[STEP {step_idx}]")
            self.log(f"Chosen Node: Node ID {n.id} | Action: {n.action} | g(n) = {n.g}, h(n) = {n.h} | f(n) = {n.f}")
            
            if n.state == goal_puzzle:
                self.log(f"[STEP LOG END] -> Optimal Goal Found at Node {n.id}!")
                path = solution(n)
                messagebox.showinfo("Thành công", f"A* đã tìm thấy lời giải tối ưu nhất sau {len(path)-1} bước!")
                return

            self.log("Children Nodes generated and evaluated:")
            for m_node in expand_8puzzle(n):
                m = m_node.state
                g_new = m_node.g
                h_m = manhattan_heuristic_8puzzle(m, goal_puzzle)
                f_m = g_new + h_m
                
                if m in reached:
                    if g_new < reached[m].g:
                        reached[m].g = g_new
                        reached[m].f = f_m
                        reached[m].parent = n 
                        heapq.heappush(frontier, (reached[m].f, reached[m]))
                        self.log(f"  - Node ID {m_node.id} | Action: {m_node.action} | g(n) = {g_new} | h(n) = {h_m} | f(n) = {f_m} (Overwrote old node with better path cost)")
                else:
                    m_node.h = h_m
                    m_node.f = f_m
                    reached[m] = m_node
                    heapq.heappush(frontier, (m_node.f, m_node))
                    self.log(f"  - Node ID {m_node.id} | Action: {m_node.action} | g(n) = {g_new} | h(n) = {h_m} | f(n) = {f_m}")
                    
        messagebox.showerror("Thất bại", "Không tìm thấy lời giải!")

if __name__ == "__main__":
    root = tk.Tk()
    app = AStarPuzzleGUI(root)
    root.mainloop()